In [71]:
import pathlib
import pandera.polars as pa
import pandera.typing.polars
from pandera.typing.polars import DataFrame
from pandera.typing.polars import Series
import polars as pl
from typing import Annotated
import matplotlib.pyplot as plt


In [72]:
## CONSTANTS
CCYS = ['jpy','chf','cad','aud','nzd','eur','gbp']
INVERSE_CCYS = ['chf','jpy','cad']


In [73]:
def get_file_path(ccy: str):
    if ccy in INVERSE_CCYS:
        pair = f'usd{ccy}'
    else:
        pair = f'{ccy}usd'
    return pathlib.Path(r'D:\Trading\2026\\',f"{pair}_d1.csv")


In [74]:
class DayBar(pa.DataFrameModel):
    timestamp: pandera.typing.polars.Series[pl.Datetime("ns", "UTC")]
    open: pandera.typing.polars.Series[float]
    high: pandera.typing.polars.Series[float]
    low: pandera.typing.polars.Series[float]
    close: pandera.typing.polars.Series[float]

In [75]:
def read_data(ccy: str) -> DataFrame[DayBar]:
    df = pl.read_csv(get_file_path(ccy))

    df = df.with_columns(
        pl.from_epoch("timestamp", time_unit="ms")
          .dt.cast_time_unit("ns")
          .dt.replace_time_zone("UTC")
    )

    if ccy in INVERSE_CCYS:
        df = df.with_columns(
            (1 / pl.col("open")).alias(f"open"),
            (1 / pl.col("high")).alias("high"),
            (1 / pl.col("low")).alias("low"),
            (1 / pl.col("close")).alias("close"),
        )
    return DayBar.validate(df)

In [76]:
bar_dict = {x: read_data(x) for x in CCYS}

In [77]:
renamed_dict = {
    ccy: bar_dict[ccy].select("timestamp",
                              pl.col("open").alias(f"{ccy}_open"),
                              pl.col("high").alias(f"{ccy}_high"),
                              pl.col("low").alias(f"{ccy}_low"),
                              pl.col("close").alias(f"{ccy}_close"))
    for ccy in CCYS
}

In [90]:
def add_return(df: pl.DataFrame, lookback: int) -> pl.DataFrame:
    df = df.with_columns(
        [(pl.col(f"{ccy}_close")/pl.col(f"{ccy}_close").shift(lookback)-1).alias(f"{ccy}_return_{lookback}") for ccy in CCYS])
    return df
git

In [ ]:
def calculate_target(df: pl.DataFrame, ccy: str) -> pl.DataFrame:
    #calculate the cross sectional 1 day return (handle multiple day look forward?
    ccy_returns = []
    relative_return =
    return relative_return

In [91]:
# Align each close series by its timestamp; retain timestamps present in any pair.
all = (
    pl.concat(renamed_dict.values(), how="align_full")
    .sort("timestamp")
).drop_nulls()
all

timestamp,jpy_open,jpy_high,jpy_low,jpy_close,chf_open,chf_high,chf_low,chf_close,cad_open,cad_high,cad_low,cad_close,aud_open,aud_high,aud_low,aud_close,nzd_open,nzd_high,nzd_low,nzd_close,eur_open,eur_high,eur_low,eur_close,gbp_open,gbp_high,gbp_low,gbp_close
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2000-03-03 00:00:00 UTC,0.00927,0.009256,0.009316,0.009283,0.601287,0.597015,0.60241,0.598086,0.686342,0.685871,0.689465,0.68975,0.6069,0.6103,0.6054,0.6083,0.4872,0.4925,0.4871,0.489,0.9648,0.9677,0.958,0.9577,1.5769,1.584,1.5754,1.5778
2000-03-04 00:00:00 UTC,0.009283,0.009283,0.009283,0.009283,0.598086,0.598086,0.598086,0.598086,0.68975,0.68975,0.68975,0.68975,0.6083,0.6083,0.6083,0.6083,0.489,0.489,0.489,0.489,0.9577,0.9577,0.9577,0.9577,1.5778,1.5778,1.5778,1.5778
2000-03-05 00:00:00 UTC,0.009283,0.009283,0.009283,0.009283,0.598086,0.598086,0.598086,0.598086,0.68975,0.68975,0.68975,0.68975,0.6083,0.6083,0.6083,0.6083,0.489,0.489,0.489,0.489,0.9577,0.9577,0.9577,0.9577,1.5778,1.5778,1.5778,1.5778
2000-03-06 00:00:00 UTC,0.009271,0.009266,0.009368,0.009299,0.596267,0.59538,0.60154,0.596623,0.689275,0.686342,0.690131,0.688326,0.6083,0.6085,0.6012,0.602,0.4884,0.4907,0.4803,0.4826,0.9574,0.967,0.9562,0.959,1.5772,1.5823,1.5713,1.5748
2000-03-07 00:00:00 UTC,0.0093,0.009248,0.009451,0.009419,0.596481,0.592909,0.598444,0.597015,0.688279,0.687144,0.689132,0.687663,0.601,0.6059,0.601,0.6047,0.4826,0.4851,0.4815,0.484,0.9585,0.9622,0.9527,0.9581,1.5745,1.5821,1.5695,1.5796
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-27 00:00:00 UTC,0.006339,0.006331,0.006355,0.006339,1.112384,1.107775,1.112545,1.10882,0.693996,0.692094,0.695691,0.694054,0.62225,0.62256,0.61999,0.62146,0.56244,0.5643,0.56135,0.56232,1.04216,1.0444,1.04049,1.04225,1.25273,1.25922,1.25046,1.25714
2024-12-28 00:00:00 UTC,0.006339,0.006339,0.006339,0.006339,1.10882,1.10882,1.10882,1.10882,0.694054,0.694054,0.694054,0.694054,0.62146,0.62146,0.62146,0.62146,0.56232,0.56232,0.56232,0.56232,1.04225,1.04225,1.04225,1.04225,1.25714,1.25714,1.25714,1.25714
2024-12-29 00:00:00 UTC,0.006342,0.006333,0.006343,0.006333,1.109619,1.108783,1.111,1.108844,0.693842,0.693727,0.694367,0.694338,0.62126,0.62339,0.62105,0.62243,0.56269,0.56462,0.56243,0.56385,1.04247,1.04346,1.04247,1.04292,1.25706,1.25876,1.25615,1.25775


In [92]:
all = add_return(all, 1)

In [93]:
all

timestamp,jpy_open,jpy_high,jpy_low,jpy_close,chf_open,chf_high,chf_low,chf_close,cad_open,cad_high,cad_low,cad_close,aud_open,aud_high,aud_low,aud_close,nzd_open,nzd_high,nzd_low,nzd_close,eur_open,eur_high,eur_low,eur_close,gbp_open,gbp_high,gbp_low,gbp_close,jpy_return_1,chf_return_1,cad_return_1,aud_return_1,nzd_return_1,eur_return_1,gbp_return_1
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2000-03-03 00:00:00 UTC,0.00927,0.009256,0.009316,0.009283,0.601287,0.597015,0.60241,0.598086,0.686342,0.685871,0.689465,0.68975,0.6069,0.6103,0.6054,0.6083,0.4872,0.4925,0.4871,0.489,0.9648,0.9677,0.958,0.9577,1.5769,1.584,1.5754,1.5778,null,null,null,null,null,null,null
2000-03-04 00:00:00 UTC,0.009283,0.009283,0.009283,0.009283,0.598086,0.598086,0.598086,0.598086,0.68975,0.68975,0.68975,0.68975,0.6083,0.6083,0.6083,0.6083,0.489,0.489,0.489,0.489,0.9577,0.9577,0.9577,0.9577,1.5778,1.5778,1.5778,1.5778,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2000-03-05 00:00:00 UTC,0.009283,0.009283,0.009283,0.009283,0.598086,0.598086,0.598086,0.598086,0.68975,0.68975,0.68975,0.68975,0.6083,0.6083,0.6083,0.6083,0.489,0.489,0.489,0.489,0.9577,0.9577,0.9577,0.9577,1.5778,1.5778,1.5778,1.5778,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2000-03-06 00:00:00 UTC,0.009271,0.009266,0.009368,0.009299,0.596267,0.59538,0.60154,0.596623,0.689275,0.686342,0.690131,0.688326,0.6083,0.6085,0.6012,0.602,0.4884,0.4907,0.4803,0.4826,0.9574,0.967,0.9562,0.959,1.5772,1.5823,1.5713,1.5748,0.001674,-0.002446,-0.002065,-0.010357,-0.013088,0.001357,-0.001901
2000-03-07 00:00:00 UTC,0.0093,0.009248,0.009451,0.009419,0.596481,0.592909,0.598444,0.597015,0.688279,0.687144,0.689132,0.687663,0.601,0.6059,0.601,0.6047,0.4826,0.4851,0.4815,0.484,0.9585,0.9622,0.9527,0.9581,1.5745,1.5821,1.5695,1.5796,0.012904,0.000657,-0.000963,0.004485,0.002901,-0.000938,0.003048
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-27 00:00:00 UTC,0.006339,0.006331,0.006355,0.006339,1.112384,1.107775,1.112545,1.10882,0.693996,0.692094,0.695691,0.694054,0.62225,0.62256,0.61999,0.62146,0.56244,0.5643,0.56135,0.56232,1.04216,1.0444,1.04049,1.04225,1.25273,1.25922,1.25046,1.25714,0.0,-0.003216,0.000069,-0.00127,-0.000213,0.000086,0.00352
2024-12-28 00:00:00 UTC,0.006339,0.006339,0.006339,0.006339,1.10882,1.10882,1.10882,1.10882,0.694054,0.694054,0.694054,0.694054,0.62146,0.62146,0.62146,0.62146,0.56232,0.56232,0.56232,0.56232,1.04225,1.04225,1.04225,1.04225,1.25714,1.25714,1.25714,1.25714,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-12-29 00:00:00 UTC,0.006342,0.006333,0.006343,0.006333,1.109619,1.108783,1.111,1.108844,0.693842,0.693727,0.694367,0.694338,0.62126,0.62339,0.62105,0.62243,0.56269,0.56462,0.56243,0.56385,1.04247,1.04346,1.04247,1.04292,1.25706,1.25876,1.25615,1.25775,-0.000849,0.000022,0.00041,0.001561,0.002721,0.000643,0.000485
